In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.ml.feature import VectorAssembler, StandardScaler


In [2]:
spark = SparkSession.builder \
    .appName("CustomerSegmentation") \
    .getOrCreate()


In [3]:
df = spark.sql("""
        SELECT customer_sk, recency, frequency, monetary 
        FROM lakehouse.gold.customer_rfm;
    """)

In [4]:
df.show(10)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

+-----------+-------+---------+--------+
|customer_sk|recency|frequency|monetary|
+-----------+-------+---------+--------+
|          1|    792|        1|  499.95|
|          2|    136|        4|  899.88|
|          3|    229|        5|  827.94|
|          4|    380|        4|  444.92|
|          5|    457|        3|  499.94|
|          6|    646|        4|  609.96|
|          7|    220|        7| 1119.84|
|          8|    126|        8| 1703.89|
|          9|    140|        5| 1224.93|
|         10|    307|        2|  274.95|
+-----------+-------+---------+--------+
only showing top 10 rows



In [4]:
df = df.filter(col("recency") < 180)  #chỉ lấy khách hàng mua trong 6 tháng

In [5]:
assembler = VectorAssembler(
    inputCols=["recency", "frequency", "monetary"],
    outputCol="rfm_features"
)
assembled_df = assembler.transform(df)

In [6]:
# Chuẩn hóa features để tránh ảnh hưởng bởi thang đo khác nhau
scaler = StandardScaler(
    inputCol="rfm_features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(assembled_df)
scaled_df = scaler_model.transform(assembled_df)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
[Stage 0:>                                                          (0 + 1) / 1]

#
# A fatal error has been detected by the Java Runtime Environment:
#
#  SIGSEGV (0xb) at pc=0x00007fad2c7e025b, pid=1438, tid=1553
#
# JRE version: OpenJDK Runtime Environment (11.0.26+4) (build 11.0.26+4-post-Debian-1deb11u1)
# Java VM: OpenJDK 64-Bit Server VM (11.0.26+4-post-Debian-1deb11u1, mixed mode, sharing, tiered, compressed oops, g1 gc, linux-amd64)
# Problematic frame:
# j  org.apache.iceberg.shaded.io.netty.util.internal.InternalThreadLocalMap.slowGet()Lorg/apache/iceberg/shaded/io/netty/util/internal/InternalThreadLocalMap;+0
#
# Core dump will be written. Default location: Core dumps may be processed with "/wsl-capture-crash %t %E %p %s" (or dumping to /src/notebooks/core.1438)
#
[thread 1470 also had an error]
# An error report file with more information is saved as:
# /src/notebooks/hs_err_pid1438.log
Compiled method (nm)   15408  220     n 0       java.lang.Thread::currentThread (native)
 total in heap  [0x00007fad3429c890,0x00007fad3429ccb0] = 1056
 relocation     [

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.p

Py4JError: An error occurred while calling o80.fit